In [2]:
!pip install statsmodels

   ---------------------------------------- 0.0/11.3 MB ? eta -:--:--
    --------------------------------------- 0.3/11.3 MB ? eta -:--:--
    --------------------------------------- 0.3/11.3 MB ? eta -:--:--
   - -------------------------------------- 0.5/11.3 MB 730.2 kB/s eta 0:00:15
   - -------------------------------------- 0.5/11.3 MB 730.2 kB/s eta 0:00:15
   -- ------------------------------------- 0.8/11.3 MB 714.3 kB/s eta 0:00:15
   --- ------------------------------------ 1.0/11.3 MB 699.0 kB/s eta 0:00:15
   --- ------------------------------------ 1.0/11.3 MB 699.0 kB/s eta 0:00:15
   ---- ----------------------------------- 1.3/11.3 MB 699.0 kB/s eta 0:00:15
   ---- ----------------------------------- 1.3/11.3 MB 699.0 kB/s eta 0:00:15
   ----- ---------------------------------- 1.6/11.3 MB 699.0 kB/s eta 0:00:14
   ------- -------------------------------- 2.1/11.3 MB 876.3 kB/s eta 0:00:11
   -------- ------------------------------- 2.4/11.3 MB 958.5 kB/s eta 0:00:10


In [3]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats
from statsmodels.stats.outliers_influence import variance_inflation_factor


In [4]:
df = pd.read_csv("insurance.csv")
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("Column names:", list(df.columns))

Rows: 1338
Columns: 7
Column names: ['age', 'sex', 'bmi', 'children', 'smoker', 'region', 'charges']


In [5]:
print("\nDESCRIPTIVE STATISTICS")

numeric_columns = df.select_dtypes(include=np.number)

descriptive_stats = pd.DataFrame({
    "Mean": numeric_columns.mean(),
    "Median": numeric_columns.median(),
    "Standard Deviation": numeric_columns.std(),
    "IQR": numeric_columns.quantile(0.75) - numeric_columns.quantile(0.25),
    "Skewness": numeric_columns.skew(),
    "Kurtosis": numeric_columns.kurtosis()
})

print(descriptive_stats.round(4))


DESCRIPTIVE STATISTICS
                Mean    Median  Standard Deviation         IQR  Skewness  \
age          39.2070    39.000             14.0500     24.0000    0.0557   
bmi          30.6634    30.400              6.0982      8.3975    0.2840   
children      1.0949     1.000              1.2055      2.0000    0.9384   
charges   13270.4223  9382.033          12110.0112  11899.6254    1.5159   

          Kurtosis  
age        -1.2451  
bmi        -0.0507  
children    0.2025  
charges     1.6063  


In [6]:
# HYPOTHESIS TEST 1
# Smokers vs Non-Smokers: Medical Charges
# H0: Medical charges do not differ between smokers and non-smokers.
# H1: Medical charges differ between smokers and non-smokers.

smokers = df[df["smoker"] == "yes"]["charges"]
non_smokers = df[df["smoker"] == "no"]["charges"]

print("\nH0: Medical charges do not differ between smokers and non-smokers.")
print("H1: Medical charges differ between smokers and non-smokers.")

# Shapiro-Wilk normality test
shapiro_smokers = stats.shapiro(smokers)
shapiro_non_smokers = stats.shapiro(non_smokers)

print("\nShapiro-Wilk Test:")
print("Smokers p-value:", shapiro_smokers.pvalue)
print("Non-smokers p-value:", shapiro_non_smokers.pvalue)

# Levene's test for equality of variances
levene_test = stats.levene(smokers, non_smokers)

print("\nLevene's Test:")
print("p-value:", levene_test.pvalue)

# Decide whether data is normal
if shapiro_smokers.pvalue > 0.05 and shapiro_non_smokers.pvalue > 0.05:

    print("\nBoth groups are normally distributed.")

    # Check equal variance
    if levene_test.pvalue > 0.05:
        print("Variances are equal.")
        print("Using Independent Two-Sample t-test.")

        test_result = stats.ttest_ind(
            smokers,
            non_smokers,
            equal_var=True
        )

    else:
        print("Variances are not equal.")
        print("Using Welch's t-test.")

        test_result = stats.ttest_ind(
            smokers,
            non_smokers,
            equal_var=False
        )

    print("Test statistic:", test_result.statistic)
    print("p-value:", test_result.pvalue)

else:

    print("\nData is not normally distributed.")
    print("Using Mann-Whitney U test.")

    test_result = stats.mannwhitneyu(
        smokers,
        non_smokers,
        alternative="two-sided"
    )

    print("Test statistic:", test_result.statistic)
    print("p-value:", test_result.pvalue)


# Conclusion
if test_result.pvalue < 0.05:
    print("Conclusion: Reject H0.")
    print("There is a statistically significant difference in medical")
    print("charges between smokers and non-smokers.")
else:
    print("Conclusion: Fail to reject H0.")
    print("There is no statistically significant difference in medical")
    print("charges between smokers and non-smokers.")

print("\nMean charges of smokers:", smokers.mean())
print("Mean charges of non-smokers:", non_smokers.mean())




H0: Medical charges do not differ between smokers and non-smokers.
H1: Medical charges differ between smokers and non-smokers.

Shapiro-Wilk Test:
Smokers p-value: 3.6249900590074145e-09
Non-smokers p-value: 1.4459006706130622e-28

Levene's Test:
p-value: 1.5593284881791394e-66

Data is not normally distributed.
Using Mann-Whitney U test.
Test statistic: 284133.0
p-value: 5.270233444503571e-130
Conclusion: Reject H0.
There is a statistically significant difference in medical
charges between smokers and non-smokers.

Mean charges of smokers: 32050.23183153284
Mean charges of non-smokers: 8434.268297856204


In [7]:
# HYPOTHESIS TEST 2A
# CHI-SQUARE: Smoking Status vs Region

# H0: Smoking status and region are independent.
# H1: Smoking status and region are associated.

print("\nH0: Smoking status and region are independent.")
print("H1: Smoking status and region are associated.")

# Create contingency table
contingency_table = pd.crosstab(
    df["smoker"],
    df["region"]
)

print("\nContingency Table:")
print(contingency_table)

# Chi-square test
chi2, chi_p, chi_dof, expected = stats.chi2_contingency(
    contingency_table
)

print("\nChi-square statistic:", chi2)
print("Degrees of freedom:", chi_dof)
print("p-value:", chi_p)

if chi_p < 0.05:
    print("Conclusion: Reject H0.")
    print("Smoking status and region are significantly associated.")
else:
    print("Conclusion: Fail to reject H0.")
    print("There is no significant association between smoking status and region.")



H0: Smoking status and region are independent.
H1: Smoking status and region are associated.

Contingency Table:
region  northeast  northwest  southeast  southwest
smoker                                            
no            257        267        273        267
yes            67         58         91         58

Chi-square statistic: 7.34347776140707
Degrees of freedom: 3
p-value: 0.06171954839170547
Conclusion: Fail to reject H0.
There is no significant association between smoking status and region.


In [8]:
# HYPOTHESIS TEST 2B
# ONE-WAY ANOVA: Charges Across Regions

# H0: Mean medical charges are equal across all regions.
# H1: At least one region has a different mean charge.

print("\nH0: Mean medical charges are equal across all regions.")
print("H1: At least one region has a different mean charge.")

# Separate charges for each region
northeast = df[df["region"] == "northeast"]["charges"]
northwest = df[df["region"] == "northwest"]["charges"]
southeast = df[df["region"] == "southeast"]["charges"]
southwest = df[df["region"] == "southwest"]["charges"]

# One-way ANOVA
anova_result = stats.f_oneway(
    northeast,
    northwest,
    southeast,
    southwest
)

print("\nF-statistic:", anova_result.statistic)
print("p-value:", anova_result.pvalue)

if anova_result.pvalue < 0.05:
    print("Conclusion: Reject H0.")
    print("There is a significant difference in mean charges")
    print("between at least two regions.")
else:
    print("Conclusion: Fail to reject H0.")
    print("There is no significant difference in mean charges across regions.")



H0: Mean medical charges are equal across all regions.
H1: At least one region has a different mean charge.

F-statistic: 2.9696266935891193
p-value: 0.030893356070497396
Conclusion: Reject H0.
There is a significant difference in mean charges
between at least two regions.


In [9]:
model = smf.ols(
    "charges ~ age + bmi + children + C(sex) + C(smoker) + C(region)",
    data=df
).fit()

print("\nRegression Model:")
print("charges ~ age + bmi + children + sex + smoker + region")



Regression Model:
charges ~ age + bmi + children + sex + smoker + region


In [10]:
print(model.summary())


                            OLS Regression Results                            
Dep. Variable:                charges   R-squared:                       0.751
Model:                            OLS   Adj. R-squared:                  0.749
Method:                 Least Squares   F-statistic:                     500.8
Date:                Tue, 08 Sep 2026   Prob (F-statistic):               0.00
Time:                        09:30:54   Log-Likelihood:                -13548.
No. Observations:                1338   AIC:                         2.711e+04
Df Residuals:                    1329   BIC:                         2.716e+04
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
                             coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------
Intercept              -1.19

In [11]:
# Regression Coefficients

coefficients = pd.DataFrame({
    "Coefficient": model.params,
    "p-value": model.pvalues,
    "CI Lower": model.conf_int()[0],
    "CI Upper": model.conf_int()[1]
})

print(coefficients.round(4))

                        Coefficient  p-value    CI Lower    CI Upper
Intercept               -11938.5386   0.0000 -13876.3934 -10000.6837
C(sex)[T.male]            -131.3144   0.6933   -784.4703    521.8416
C(smoker)[T.yes]         23848.5345   0.0000  23038.0307  24659.0384
C(region)[T.northwest]    -352.9639   0.4588  -1287.2982    581.3704
C(region)[T.southeast]   -1035.0220   0.0308  -1974.0968    -95.9473
C(region)[T.southwest]    -960.0510   0.0448  -1897.6364    -22.4656
age                        256.8564   0.0000    233.5138    280.1989
bmi                        339.1935   0.0000    283.0884    395.2985
children                   475.5005   0.0006    205.1633    745.8378


In [ ]:
# R-Squared

print("R-squared:", round(model.rsquared, 4))
print("Adjusted R-squared:", round(model.rsquared_adj, 4))



------------------------------------------------------------
MODEL PERFORMANCE
------------------------------------------------------------
R-squared: 0.7509
Adjusted R-squared: 0.7494


In [13]:
# Diagnostic Tests

residuals = model.resid

In [14]:
# Jarque-Bera Test
jb_stat, jb_pvalue = stats.jarque_bera(residuals)

print("\nJarque-Bera Test")
print("Statistic:", jb_stat)
print("p-value:", jb_pvalue)


# Omnibus Test
omnibus_stat, omnibus_pvalue = sm.stats.omni_normtest(residuals)

print("\nOmnibus Test")
print("Statistic:", omnibus_stat)
print("p-value:", omnibus_pvalue)


Jarque-Bera Test
Statistic: 718.8872635707893
p-value: 7.86346866182781e-157

Omnibus Test
Statistic: 300.36591553235564
p-value: 5.975443801310551e-66


In [15]:
# VIF - Multicollinearity

print("\n" + "-" * 60)
print("VIF FOR CONTINUOUS PREDICTORS")
print("-" * 60)

X = df[["age", "bmi", "children"]]

# Add constant
X = sm.add_constant(X)

vif_table = pd.DataFrame()

vif_table["Feature"] = X.columns
vif_table["VIF"] = [
    variance_inflation_factor(X.values, i)
    for i in range(X.shape[1])
]

# Remove constant row
vif_table = vif_table[vif_table["Feature"] != "const"]

print(vif_table.round(4))


------------------------------------------------------------
VIF FOR CONTINUOUS PREDICTORS
------------------------------------------------------------
    Feature     VIF
1       age  1.0138
2       bmi  1.0122
3  children  1.0019
